# Valkey Search Workshop — Streaming Service Demo

## 3-Hour Hands-On Workshop

In this workshop you'll build a movie recommendation system using **Valkey Search**, learning:

1. **Index creation** — TEXT, TAG, NUMERIC, VECTOR fields
2. **FT.SEARCH** — Full-text search, tag filters, numeric ranges
3. **Vector similarity (KNN)** — Find similar movies using embeddings
4. **Hybrid search** — Combine filters with vector similarity
5. **Single-slot indexes** — Micro-latency per-user queries
6. **FT.AGGREGATE** — Platform analytics (trending, most watched)
7. **Cross-index workflows** — Connecting user history to catalog recommendations

### Architecture

| Index | Purpose | Latency |
|:------|:--------|:--------|
| `idx:movies` (Global) | Movie catalog with vectors | Micro-ms |
| `idx:user:<id>:history` (Single-Slot) | One user's watch history | Sub-ms |
| `idx:watch` (Global) | All users' history for analytics | Micro-ms |

### Dataset
- **6,300 movies** with 768-dim embeddings (from TMDB)
- **60,000 ratings** from 500 users (from MovieLens)


## Part 1: Setup & Connection (~15 min)

### Option A: Google Colab
Run the cell below to install Valkey with the Search module directly in Colab.

### Option B: Local Machine
Run `docker compose up -d` in the workshop directory, then skip the install cell.


In [1]:
# === SETUP ===
# Starts Valkey with Search module using Docker or Podman.
# Works on Windows (Podman/Docker) and Mac (Podman/Colima/Docker).

import subprocess, socket, time, shutil

# Detect container runtime
runtime = 'podman' if shutil.which('podman') else 'docker'
print(f'Using: {runtime}')

# Stop and remove any existing Valkey container
subprocess.run([runtime, 'rm', '-f', 'valkey'], capture_output=True)
time.sleep(1)

# Start Valkey
subprocess.run([runtime, 'run', '-d', '--name', 'valkey', '-p', '6379:6379',
    'valkey/valkey-bundle:9.1.0-rc2',
    'valkey-server', '--save', '', '--protected-mode', 'no'])

print('Waiting for Valkey to start...')
time.sleep(5)

# Verify by pinging inside the container (bypasses Windows networking issues)
result = subprocess.run([runtime, 'exec', 'valkey', 'valkey-cli', 'ping'],
    capture_output=True, text=True)
assert 'PONG' in result.stdout, f'Failed to start. Check: {runtime} logs valkey'
print('✅ Valkey started')


Using: docker
cd82c9377a89b27bed9b054864223c468f99eafe97cdbdbbaba2834ea2699fd0
Waiting for Valkey to start...
✅ Valkey started


> **Prerequisites**: Docker installed and running.  
> If the cell above fails, run manually:  
> ```
> docker run -d --name valkey -p 6379:6379 valkey/valkey-bundle:9.1.0-rc2 valkey-server --save "" --protected-mode no
> ```

In [2]:
%pip install -q valkey pandas numpy


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
required = ['catalog.csv', 'movies.csv', 'ratings.csv', 'users.txt']
missing = [f for f in required if not os.path.exists(f'data/{f}')]
if missing:
    raise FileNotFoundError(f'Missing data files: {missing}. Upload the data/ folder.')
print(f'✅ Data ready: {os.listdir("data")}')

✅ Data ready: ['movies.csv', 'ratings.csv', 'users.txt', 'catalog.csv']


In [4]:
import csv, struct, time, subprocess, json
import pandas as pd
import numpy as np
import valkey

# Connect to Valkey — handle Podman Windows networking
VALKEY_HOST = 'localhost'
VALKEY_PORT = 6379

try:
    r = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=3)
    r.ping()
except Exception:
    # Podman on Windows: localhost may not route. Get container IP instead.
    inspect = subprocess.run([runtime, 'inspect', 'valkey'], capture_output=True, text=True)
    info = json.loads(inspect.stdout)
    VALKEY_HOST = info[0]['NetworkSettings']['IPAddress'] or 'localhost'
    print(f'localhost failed, using container IP: {VALKEY_HOST}')
    r = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=3)

r_bin = valkey.Valkey(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=False, socket_timeout=3)

print(f'Connected: {r.ping()}')
modules = r_bin.module_list()
module_names = [m[b'name'].decode() for m in modules]
print(f'Modules: {module_names}')
assert 'search' in module_names, 'ERROR: search module not loaded!'
print('✅ Ready!')


Connected: True
Modules: ['search', 'json', 'bf', 'ldap', 'lua']
✅ Ready!


In [5]:
# Explore the datasets
import pandas as pd

# --- Data Sources ---
# Two original sources:
#   TMDB (via HuggingFace) → overview, vote_average, popularity, language, 768-dim vectors
#   MovieLens (ml-32m)     → user ratings, movie titles, genres, movieId↔tmdbId mapping
#
# --- Workshop CSVs ---
#
# catalog.csv  | Source: TMDB (HuggingFace parquet)
#              | Created: 6,300 rows extracted from the full 851K TMDB catalog (full dataset), matching tmdbIds
#              |          of movies our 500 users watched. Vectors are 768-dim PCA on
#              |          TMDB text embeddings (title+genres+overview).
#              | Used for: global catalog (title, overview, genres, vector)
#              | Index: idx:movies
#
# ratings.csv  | Source: MovieLens 32M
#              | Created: subset of ml-32m/ratings.csv — 500 users with 50-200 ratings
#              |          each (60,000 rows out of 32M).
#              | Used for: user ratings, joined with movies.csv for both indexes below
#              | Index: idx:user:<id>:history + idx:watch
#
# movies.csv   | Source: MovieLens 32M + ml-32m/links.csv
#              | Created: subset of ml-32m/movies.csv merged with links.csv to add
#              |          tmdbId. Only the 6,300 movies our 500 users rated.
#              | Used for: provides title, genres, tmdbId for each rating
#              | Index: idx:user:<id>:history + idx:watch
#
# users.txt    | 500 user IDs picked from ratings.csv

print('=== catalog.csv (from TMDB — global catalog) ===')
catalog = pd.read_csv('data/catalog.csv', encoding='utf-8', nrows=3)
print(f'Rows: {sum(1 for _ in open("data/catalog.csv", encoding="utf-8")) - 1} movies')
print(f'Columns: {list(catalog.columns)}')
print(catalog.drop(columns=['vector', 'overview']).to_string(index=False))

print(f'\n=== ratings.csv (from MovieLens — user ratings) ===')
ratings = pd.read_csv('data/ratings.csv', encoding='utf-8')
print(f'Rows: {len(ratings)} ratings | Users: {ratings["userId"].nunique()} | Movies: {ratings["movieId"].nunique()}')
print(f'Rating range: {ratings["rating"].min()} – {ratings["rating"].max()}')
print(ratings.head(3).to_string(index=False))

print(f'\n=== movies.csv (from MovieLens — joins ratings to catalog via tmdbId) ===')
movies = pd.read_csv('data/movies.csv', encoding='utf-8')
print(f'Rows: {len(movies)} | Columns: {list(movies.columns)}')
print(movies.head(3).to_string(index=False))


=== catalog.csv (from TMDB — global catalog) ===
Rows: 6304 movies
Columns: ['id', 'title', 'overview', 'genres', 'vote_average', 'popularity', 'original_language', 'vector']
 id          title                        genres  vote_average  popularity original_language
  2          Ariel Comedy, Drama, Romance, Crime           7.1      11.915                fi
  5     Four Rooms                        Comedy           5.9      24.557                en
  6 Judgment Night       Action, Crime, Thriller           6.5      12.110                en

=== ratings.csv (from MovieLens — user ratings) ===
Rows: 60601 ratings | Users: 500 | Movies: 6347
Rating range: 0.5 – 5.0
 userId  movieId  rating  timestamp
      1       17     4.0  944249077
      1       25     1.0  944250228
      1       29     2.0  943230976

=== movies.csv (from MovieLens — joins ratings to catalog via tmdbId) ===
Rows: 6343 | Columns: ['movieId', 'title', 'genres', 'tmdbId']
 movieId                   title              

## Part 2: Global Catalog Index (~45 min)

A **global index** is distributed across all shards in a cluster.
- Each shard holds a portion of the documents
- Queries fan out to every shard, results are merged and returned

**Schema design choices:**
- `title`, `overview` → **TEXT**: tokenized for keyword search and prefix matching
- `genres`, `original_language` → **TAG**: discrete values, exact/set matching, not tokenized
- `vote_average`, `popularity` → **NUMERIC SORTABLE**: range filters and sort ordering
- `vector` → **VECTOR FLAT**: similarity search via KNN (FLAT for small datasets, HNSW for large)

We load documents first (plain HSET), then create the index. Valkey backfills existing keys asynchronously.

### 2.1 Load Data


In [6]:
# Example: what one document looks like in the catalog
# Each movie becomes a HASH key: movie:<tmdb_id>
print('''
Key:    movie:11
Fields:
  title             = "Star Wars"                    (TEXT — full-text searchable)
  overview          = "Princess Leia is captured..." (TEXT — searchable description)
  genres            = "Adventure,Action,Sci-Fi"      (TAG — comma-separated, filterable)
  original_language = "en"                           (TAG — exact match filter)
  vote_average      = 8.2                            (NUMERIC — range queries, sortable)
  popularity        = 101.1                          (NUMERIC — range queries, sortable)
  vector            = <3072 bytes>                   (VECTOR — 768 × float32, for KNN)
''')



Key:    movie:11
Fields:
  title             = "Star Wars"                    (TEXT — full-text searchable)
  overview          = "Princess Leia is captured..." (TEXT — searchable description)
  genres            = "Adventure,Action,Sci-Fi"      (TAG — comma-separated, filterable)
  original_language = "en"                           (TAG — exact match filter)
  vote_average      = 8.2                            (NUMERIC — range queries, sortable)
  popularity        = 101.1                          (NUMERIC — range queries, sortable)
  vector            = <3072 bytes>                   (VECTOR — 768 × float32, for KNN)



In [7]:
# Start fresh
r.flushall()

# Load movies from CSV with binary-packed vectors
count = 0
pipe = r_bin.pipeline(transaction=False)
t0 = time.time()

with open('data/catalog.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Pack 768 floats into binary blob (required for vector indexing)
        vec_floats = [float(x) for x in row['vector'].split(',')]
        vec_blob = struct.pack(f'{len(vec_floats)}f', *vec_floats)
        
        pipe.hset(f'movie:{row["id"]}'.encode(), mapping={
            b'title': row['title'].encode(),
            b'overview': row['overview'].encode(),
            b'genres': row['genres'].encode(),
            b'original_language': row['original_language'].encode(),
            b'vote_average': row['vote_average'].encode(),
            b'popularity': row['popularity'].encode(),
            b'vector': vec_blob,
        })
        count += 1
        if count % 500 == 0:
            pipe.execute()
            pipe = r_bin.pipeline(transaction=False)

pipe.execute()
elapsed = time.time() - t0
print(f"✓ Loaded {count} movies in {elapsed:.1f}s ({count/elapsed:.0f} docs/sec)")

✓ Loaded 6304 movies in 1.3s (5002 docs/sec)


### 2.2 Create the Index

Now create the index — Valkey backfills existing keys asynchronously.


In [8]:
# Drop if exists
try:
    r.execute_command('FT.DROPINDEX', 'idx:movies')
except:
    pass

# Create the global catalog index
r.execute_command('FT.CREATE', 'idx:movies', 'ON', 'HASH', 'PREFIX', '1', 'movie:',
    'SCHEMA',
    'title', 'TEXT',                          # Full-text searchable
    'overview', 'TEXT',                       # Movie description
    'genres', 'TAG', 'SEPARATOR', ',',        # Filterable categories
    'original_language', 'TAG',               # Language filter
    'vote_average', 'NUMERIC', 'SORTABLE',    # Rating 0-10
    'popularity', 'NUMERIC', 'SORTABLE',      # Popularity score
    'vector', 'VECTOR', 'FLAT', '6',          # Vector similarity
        'TYPE', 'FLOAT32', 'DIM', '768', 'DISTANCE_METRIC', 'COSINE')

print("✓ Index idx:movies created")

✓ Index idx:movies created


In [9]:
# Wait for backfill to complete
import time
while True:
    info = r.execute_command('FT.INFO', 'idx:movies')
    info_dict = dict(zip(info[::2], info[1::2]))
    pct = float(info_dict.get('backfill_complete_percent', 1.0))
    docs = info_dict.get('num_docs', 0)
    if pct >= 1.0:
        break
    print(f'  Indexing... {pct*100:.0f}% ({docs} docs)', end='\r')
    time.sleep(1)
print(f'✓ Index ready: {info_dict.get("num_docs", 0)} docs indexed')

✓ Index ready: 6304 docs indexed


In [10]:
# Inspect the index
r.execute_command('FT.INFO', 'idx:movies')


['index_name',
 'idx:movies',
 'index_definition',
 ['key_type', 'HASH', 'prefixes', ['movie:'], 'default_score', '1'],
 'attributes',
 [['identifier',
   'genres',
   'attribute',
   'genres',
   'user_indexed_memory',
   127768,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '6302'],
  ['identifier',
   'original_language',
   'attribute',
   'original_language',
   'user_indexed_memory',
   12608,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '6304'],
  ['identifier',
   'overview',
   'attribute',
   'overview',
   'user_indexed_memory',
   1775610,
   'type',
   'TEXT',
   'WITH_SUFFIX_TRIE',
   '0',
   'NO_STEM',
   '0',
   'WEIGHT',
   '1'],
  ['identifier',
   'title',
   'attribute',
   'title',
   'user_indexed_memory',
   99246,
   'type',
   'TEXT',
   'WITH_SUFFIX_TRIE',
   '0',
   'NO_STEM',
   '0',
   'WEIGHT',
   '1'],
  ['identifier',
   'popularity',
   'attribute',
   'popularity',
   'u

### 2.3 Full-Text Search (FT.SEARCH)

Search movie titles and overviews using natural language queries.


In [11]:
# Simple text search
results = r.execute_command('FT.SEARCH', 'idx:movies', 'space adventure',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

results  # Raw response: [total_matches, key1, [field, val, ...], key2, [...], ...]

[3,
 'movie:157336',
 ['title', 'Interstellar', 'genres', 'Adventure, Drama, Science Fiction'],
 'movie:6795',
 ['title',
  'Zathura: A Space Adventure',
  'genres',
  'Science Fiction, Adventure, Family'],
 'movie:10681',
 ['title', 'WALL·E', 'genres', 'Animation, Family, Science Fiction']]

In [12]:
# Search within a specific field
r.execute_command('FT.SEARCH', 'idx:movies', '@title:Matrix',
    'RETURN', '2', 'title', 'vote_average',
    'LIMIT', '0', '5')

[4,
 'movie:604',
 ['title', 'The Matrix Reloaded', 'vote_average', '7.054'],
 'movie:603',
 ['title', 'The Matrix', 'vote_average', '8.22'],
 'movie:624860',
 ['title', 'The Matrix Resurrections', 'vote_average', '6.396'],
 'movie:605',
 ['title', 'The Matrix Revolutions', 'vote_average', '6.731']]

### 2.4 Instant Search & Title Suggestions

Prefix search enables autocomplete / type-ahead as the user types.


In [13]:
# Prefix search: autocomplete as user types "star"
r.execute_command('FT.SEARCH', 'idx:movies', '@title:star*',
    'RETURN', '1', 'title',
    'LIMIT', '0', '5')


[41,
 'movie:209276',
 ['title', 'Starred Up'],
 'movie:152',
 ['title', 'Star Trek: The Motion Picture'],
 'movie:157',
 ['title', 'Star Trek III: The Search for Spock'],
 'movie:1895',
 ['title', 'Star Wars: Episode III - Revenge of the Sith'],
 'movie:154',
 ['title', 'Star Trek II: The Wrath of Khan']]

In [14]:
# Prefix on longer input: "inter"
r.execute_command('FT.SEARCH', 'idx:movies', '@title:inter*',
    'RETURN', '1', 'title',
    'LIMIT', '0', '5')


[14,
 'movie:157336',
 ['title', 'Interstellar'],
 'movie:257211',
 ['title', 'The Intern'],
 'movie:228967',
 ['title', 'The Interview'],
 'movie:20312',
 ['title', 'Interstate 60'],
 'movie:816',
 ['title', 'Austin Powers: International Man of Mystery']]

### 2.4 Tag & Numeric Filters

TAG fields support exact-match filtering. NUMERIC fields support range queries.


In [15]:
# Tag filter: exact match on genre
r.execute_command('FT.SEARCH', 'idx:movies', '@genres:{Action}',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

[1271,
 'movie:903',
 ['title', 'Cool Hand Luke', 'genres', 'Action, Drama, Crime'],
 'movie:14282',
 ['title', 'Ninja Scroll', 'genres', 'Fantasy, Adventure, Animation, Action'],
 'movie:23834',
 ['title', 'Volcano High', 'genres', 'Action, Comedy'],
 'movie:1428',
 ['title', 'Once Upon a Time in Mexico', 'genres', 'Action, Drama, Mystery'],
 'movie:2019',
 ['title', 'Hard Target', 'genres', 'Action, Adventure, Crime, Thriller']]

In [16]:
# Combined: tag + numeric range + sort
r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Action} @vote_average:[8 10]',
    'SORTBY', 'popularity', 'DESC',
    'RETURN', '3', 'title', 'vote_average', 'popularity',
    'LIMIT', '0', '5')

[34,
 'movie:98',
 ['title', 'Gladiator', 'vote_average', '8.2', 'popularity', '405.13'],
 'movie:569094',
 ['title',
  'Spider-Man: Across the Spider-Verse',
  'vote_average',
  '8.356',
  'popularity',
  '221.902'],
 'movie:634649',
 ['title',
  'Spider-Man: No Way Home',
  'vote_average',
  '8.0',
  'popularity',
  '199.379'],
 'movie:299536',
 ['title',
  'Avengers: Infinity War',
  'vote_average',
  '8.242',
  'popularity',
  '170.118'],
 'movie:155',
 ['title', 'The Dark Knight', 'vote_average', '8.5', 'popularity', '152.084']]

In [17]:
# Multi-tag OR: Comedy | Drama, rated 8.5+
r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Comedy | Drama} @vote_average:[8.5 10]',
    'RETURN', '2', 'title', 'genres',
    'LIMIT', '0', '5')

[12,
 'movie:664280',
 ['title',
  'David Attenborough: A Life on Our Planet',
  'genres',
  'Documentary, Drama'],
 'movie:346',
 ['title', 'Seven Samurai', 'genres', 'Action, Drama'],
 'movie:13',
 ['title', 'Forrest Gump', 'genres', 'Comedy, Drama, Romance'],
 'movie:389',
 ['title', '12 Angry Men', 'genres', 'Drama'],
 'movie:372058',
 ['title', 'Your Name.', 'genres', 'Animation, Romance, Drama']]

### 2.5 Vector Similarity Search (KNN)

Each movie has a 768-dimensional embedding vector. We can find similar movies
using K-Nearest Neighbors (KNN) search with cosine distance.


In [18]:
# Get a movie's vector and find similar ones
source = r_bin.hgetall(b'movie:11')  # Star Wars (tmdb_id: 11)
print(f"Source movie: {source[b'title'].decode()}")
print(f"Vector size: {len(source[b'vector'])} bytes ({len(source[b'vector'])//4} floats)")

# KNN search — find 6 most similar (first is always self, so we skip it)
results = r_bin.execute_command('FT.SEARCH', 'idx:movies',
    '*=>[KNN 6 @vector $query_vec]',
    'PARAMS', '2', 'query_vec', source[b'vector'],
    'RETURN', '2', 'title', 'genres',
    'DIALECT', '2')

print()
print('Movies similar to Star Wars:')
for i in range(1, len(results), 2):
    if results[i] == b'movie:11':
        continue
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc[b'title'].decode()} ({doc[b'genres'].decode()})")


Source movie: Star Wars
Vector size: 3072 bytes (768 floats)

Movies similar to Star Wars:
  The Empire Strikes Back (Adventure, Action, Science Fiction)
  Return of the Jedi (Adventure, Action, Science Fiction)
  Star Wars: The Clone Wars (Animation, Action, Science Fiction, Adventure)
  Star Wars: Episode I - The Phantom Menace (Adventure, Action, Science Fiction)
  Star Wars: Episode II - Attack of the Clones (Adventure, Action, Science Fiction)


### 2.6 Hybrid Search (Filter + Vector)

Combine tag/numeric filters with vector similarity — e.g., "Action movies similar to Star Wars".


In [19]:
# Hybrid: tag filter + vector KNN
# "Action movies similar to Star Wars"
source_vec = r_bin.hget(b'movie:11', b'vector')

results = r_bin.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Action}=>[KNN 11 @vector $query_vec]',
    'PARAMS', '2', 'query_vec', source_vec,
    'RETURN', '3', 'title', 'genres', 'vote_average',
    'DIALECT', '2')

print('Action movies similar to Star Wars:')
for i in range(1, len(results), 2):
    if results[i] == b'movie:11':
        continue
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    title = doc[b'title'].decode()
    rating = doc[b'vote_average'].decode()
    print(f'  {title} ({rating}\u2605)')


Action movies similar to Star Wars:
  The Empire Strikes Back (8.393★)
  Return of the Jedi (7.899★)
  Star Wars: The Clone Wars (6.1★)
  Star Wars: Episode I - The Phantom Menace (6.6★)
  Star Wars: Episode II - Attack of the Clones (6.569★)
  Star Wars: Episode III - Revenge of the Sith (7.436★)
  Sky Captain and the World of Tomorrow (5.9★)
  Solo (4.6★)
  Krull (6.0★)


## Part 3: Single-Slot Index — Per-User History (~30 min)

### Cluster Topology: Global vs Single-Slot

In a Valkey cluster, keys are distributed across shards by hash slot.

**Global index** (Part 2): keys spread across all shards. Every query fans out.

**Single-slot index**: all keys share the same hash slot → same shard.
- Query executes locally on one shard — no fanout, no merge
- Micro latency
- One index per user — ephemeral (create on login, drop on logout)
- Achieved by key prefix: `user:1:watch:*` all hash to the same slot

```
┌──────────┐  ┌──────────┐  ┌──────────┐
│ Shard 1  │  │ Shard 2  │  │ Shard 3  │
│          │  │          │  │          │
│ movie:*  │  │ movie:*  │  │ movie:*  │  ← Global: partitioned
│          │  │          │  │          │
│user:1:*  │  │          │  │user:2:*  │  ← Single-slot: pinned
└──────────┘  └──────────┘  └──────────┘
```

### 3.1 Load & Create Per-User Indexes


In [20]:
# Example: what one document looks like in a user's watch history
# Each rating becomes a HASH key: user:<id>:watch:<n>
print('''
Key:    user:1:watch:0
Fields:
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — filterable)
  rating    = 4.0                   (NUMERIC — user's rating, 0.5-5.0)
  timestamp = 944249077             (NUMERIC — when they rated it)
  tmdb_id   = "862"                 (TAG — links to movie:862 in global catalog)
''')



Key:    user:1:watch:0
Fields:
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — filterable)
  rating    = 4.0                   (NUMERIC — user's rating, 0.5-5.0)
  timestamp = 944249077             (NUMERIC — when they rated it)
  tmdb_id   = "862"                 (TAG — links to movie:862 in global catalog)



In [21]:
# Load user data
movies_df = pd.read_csv('data/movies.csv')
ratings_df = pd.read_csv('data/ratings.csv')

# Pick 3 users for the demo
demo_users = [int(x) for x in open('data/users.txt').read().split()[:3]]
print(f"Demo users: {demo_users}")

for user_id in demo_users:
    user_ratings = ratings_df[ratings_df['userId'] == user_id].merge(movies_df, on='movieId')
    user_ratings['genres'] = user_ratings['genres'].str.replace('|', ',')
    
    idx_name = f"idx:user:{user_id}:history"
    prefix = f"user:{user_id}:watch:"
    
    # Load data FIRST
    pipe = r.pipeline(transaction=False)
    for i, (_, row) in enumerate(user_ratings.iterrows()):
        tmdb_id = str(int(row['tmdbId'])) if pd.notna(row.get('tmdbId')) else ''
        pipe.hset(f"{prefix}{i}", mapping={
            'title': str(row['title']),
            'genres': str(row['genres']),
            'rating': str(row['rating']),
            'timestamp': str(int(row['timestamp'])),
            'tmdb_id': tmdb_id,
        })
    pipe.execute()
    
    # THEN create index (backfill async)
    try:
        r.execute_command('FT.DROPINDEX', idx_name)
    except:
        pass
    r.execute_command('FT.CREATE', idx_name, 'ON', 'HASH',
        'PREFIX', '1', prefix,
        'SCHEMA',
        'title', 'TEXT',
        'genres', 'TAG', 'SEPARATOR', ',',
        'rating', 'NUMERIC', 'SORTABLE',
        'timestamp', 'NUMERIC', 'SORTABLE',
        'tmdb_id', 'TAG')
    
    print(f"  User {user_id}: {len(user_ratings)} movies loaded + indexed")

print("\n✓ Single-slot indexes created")

Demo users: [1, 2, 3]
  User 1: 141 movies loaded + indexed
  User 2: 52 movies loaded + indexed
  User 3: 147 movies loaded + indexed

✓ Single-slot indexes created


In [22]:
# Inspect one user's index
r.execute_command('FT.INFO', f'idx:user:{demo_users[0]}:history')


['index_name',
 'idx:user:1:history',
 'index_definition',
 ['key_type', 'HASH', 'prefixes', ['user:1:watch:'], 'default_score', '1'],
 'attributes',
 [['identifier',
   'rating',
   'attribute',
   'rating',
   'user_indexed_memory',
   0,
   'type',
   'NUMERIC',
   'size',
   '0'],
  ['identifier',
   'timestamp',
   'attribute',
   'timestamp',
   'user_indexed_memory',
   0,
   'type',
   'NUMERIC',
   'size',
   '0'],
  ['identifier',
   'genres',
   'attribute',
   'genres',
   'user_indexed_memory',
   0,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '0'],
  ['identifier',
   'title',
   'attribute',
   'title',
   'user_indexed_memory',
   0,
   'type',
   'TEXT',
   'WITH_SUFFIX_TRIE',
   '0',
   'NO_STEM',
   '0',
   'WEIGHT',
   '1'],
  ['identifier',
   'tmdb_id',
   'attribute',
   'tmdb_id',
   'user_indexed_memory',
   0,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '0']],
 'num_docs',
 0

### 3.2 Query User History

In [36]:
user_id = demo_users[0]
idx = f"idx:user:{user_id}:history"

# Recent watches sorted by time
print(f'--- User {user_id}: Recent watches ---')
results = r.execute_command('FT.SEARCH', idx, '@timestamp:[-inf +inf]',
    'SORTBY', 'timestamp', 'DESC',
    'RETURN', '3', 'title', 'rating', 'timestamp',
    'LIMIT', '0', '5')
print(f'Total: {results[0]} movies')
for i in range(1, len(results), 2):
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc['title']} - {doc['rating']}\u2605")


--- User 1: Recent watches ---
Total: 141 movies
  Dangerous Liaisons (1988) - 5.0★
  All About Eve (1950) - 5.0★
  Citizen Ruth (1996) - 4.0★
  Ever After: A Cinderella Story (1998) - 4.0★
  Primary Colors (1998) - 3.0★


In [37]:
# Top-rated Action movies for this user
print(f'--- User {user_id}: Top Action movies ---')
results = r.execute_command('FT.SEARCH', idx,
    '@genres:{Action} @rating:[4 5]',
    'SORTBY', 'rating', 'DESC',
    'RETURN', '2', 'title', 'rating',
    'LIMIT', '0', '5')
print(f'{results[0]} matches')
for i in range(1, len(results), 2):
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc['title']} - {doc['rating']}\u2605")


--- User 1: Top Action movies ---
14 matches
  Apocalypse Now (1979) - 5.0★
  Aliens (1986) - 5.0★
  North by Northwest (1959) - 5.0★
  Boot, Das (Boat, The) (1981) - 5.0★
  RoboCop (1987) - 5.0★


In [38]:
# Text search user's history: "Have I watched Star Wars?"
print(f'--- User {user_id}: Have I watched Star Wars? ---')
results = r.execute_command('FT.SEARCH', idx, '@title:Star Wars',
    'RETURN', '2', 'title', 'rating',
    'LIMIT', '0', '5')
if results[0] > 0:
    for i in range(1, len(results), 2):
        doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
        print(f"  Yes! {doc['title']} - rated {doc['rating']}\u2605")
else:
    print('  No - not watched yet.')


--- User 1: Have I watched Star Wars? ---
  Yes! Star Wars: Episode VI - Return of the Jedi (1983) - rated 2.0★
  Yes! Star Wars: Episode IV - A New Hope (1977) - rated 5.0★
  Yes! Star Wars: Episode V - The Empire Strikes Back (1980) - rated 5.0★


## Part 4: Global User Index & FT.AGGREGATE (~30 min)

A **second global index** over ALL users' watch history.
- Distributed across shards (keys have no hash tag)
- Enables cross-user analytics: trending, most watched, avg ratings
- Queried with `FT.AGGREGATE` — server-side GROUPBY, REDUCE, SORT
- Single-slot answers "what did *I* watch?" — this answers "what is *everyone* watching?"

### 4.1 Load & Create


In [26]:
# Example: what one document looks like in the global watch index
# Each rating becomes a HASH key: watch:<userId>:<tmdbId>
print('''
Key:    watch:1:862
Fields:
  user_id   = "1"                   (TAG — which user)
  tmdb_id   = "862"                 (TAG — which movie, links to catalog)
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — for genre analytics)
  rating    = 4.0                   (NUMERIC — for AVG/SUM aggregations)
  timestamp = 944249077             (NUMERIC — for time-based trending)
''')



Key:    watch:1:862
Fields:
  user_id   = "1"                   (TAG — which user)
  tmdb_id   = "862"                 (TAG — which movie, links to catalog)
  title     = "Toy Story"           (TEXT — searchable)
  genres    = "Adventure,Animation" (TAG — for genre analytics)
  rating    = 4.0                   (NUMERIC — for AVG/SUM aggregations)
  timestamp = 944249077             (NUMERIC — for time-based trending)



In [27]:
# Load ALL users' ratings FIRST
all_ratings = ratings_df.merge(movies_df, on='movieId')
all_ratings['genres'] = all_ratings['genres'].str.replace('|', ',')

pipe = r.pipeline(transaction=False)
count = 0
for _, row in all_ratings.iterrows():
    uid = str(int(row['userId']))
    tid = str(int(row['tmdbId'])) if pd.notna(row.get('tmdbId')) else ''
    if not tid:
        continue
    pipe.hset(f"watch:{uid}:{tid}", mapping={
        'user_id': uid,
        'tmdb_id': tid,
        'title': str(row['title']),
        'genres': str(row['genres']),
        'rating': str(row['rating']),
        'timestamp': str(int(row['timestamp'])),
    })
    count += 1
    if count % 500 == 0:
        pipe.execute()
        pipe = r.pipeline(transaction=False)
pipe.execute()
print(f"✓ Loaded {count} watch events")

# THEN create index (backfill async)
try:
    r.execute_command('FT.DROPINDEX', 'idx:watch')
except:
    pass

r.execute_command('FT.CREATE', 'idx:watch', 'ON', 'HASH', 'PREFIX', '1', 'watch:',
    'SCHEMA',
    'user_id', 'TAG',
    'tmdb_id', 'TAG',
    'title', 'TEXT',
    'genres', 'TAG', 'SEPARATOR', ',',
    'rating', 'NUMERIC', 'SORTABLE',
    'timestamp', 'NUMERIC', 'SORTABLE')

# Wait for backfill
import time
while True:
    info = r.execute_command('FT.INFO', 'idx:watch')
    d = dict(zip(info[::2], info[1::2]))
    if float(d.get('backfill_complete_percent', 1.0)) >= 1.0:
        break
    time.sleep(0.5)
print(f"✓ idx:watch ready: {d.get('num_docs', 0)} docs indexed")

✓ Loaded 60594 watch events
✓ idx:watch ready: 60594 docs indexed


In [28]:
# Inspect the global user index
r.execute_command('FT.INFO', 'idx:watch')


['index_name',
 'idx:watch',
 'index_definition',
 ['key_type', 'HASH', 'prefixes', ['watch:'], 'default_score', '1'],
 'attributes',
 [['identifier',
   'tmdb_id',
   'attribute',
   'tmdb_id',
   'user_indexed_memory',
   240475,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '60594'],
  ['identifier',
   'genres',
   'attribute',
   'genres',
   'user_indexed_memory',
   1215385,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '60594'],
  ['identifier',
   'title',
   'attribute',
   'title',
   'user_indexed_memory',
   1519898,
   'type',
   'TEXT',
   'WITH_SUFFIX_TRIE',
   '0',
   'NO_STEM',
   '0',
   'WEIGHT',
   '1'],
  ['identifier',
   'user_id',
   'attribute',
   'user_id',
   'user_indexed_memory',
   174188,
   'type',
   'TAG',
   'SEPARATOR',
   ',',
   'CASESENSITIVE',
   '0',
   'size',
   '60594'],
  ['identifier',
   'rating',
   'attribute',
   'rating',
   'user_indexed_memory',
   18

### 4.2 FT.AGGREGATE — Platform Analytics

In [29]:
# FT.AGGREGATE: Most watched movies across all users
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '1', '@tmdb_id',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'COUNT', '0', 'AS', 'watch_count',
    'SORTBY', '2', '@watch_count', 'DESC',
    'LIMIT', '0', '10')

# Raw response
print("Raw:", results[:3], "...\n")

# Server computed the counts — we just look up titles
print("Most Watched:")
for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    title = r.hget(f"movie:{doc['tmdb_id']}", 'title') or doc['tmdb_id']
    print(f"  {title} — {doc['watch_count']} watches")

Raw: [10, ['tmdb_id', '13', 'watch_count', '286'], ['tmdb_id', '680', 'watch_count', '271']] ...

Most Watched:
  Forrest Gump — 286 watches
  Pulp Fiction — 271 watches
  The Shawshank Redemption — 262 watches
  The Matrix — 257 watches
  The Silence of the Lambs — 240 watches
  Star Wars — 236 watches
  Jurassic Park — 211 watches
  The Empire Strikes Back — 210 watches
  Terminator 2: Judgment Day — 200 watches
  The Lord of the Rings: The Fellowship of the Ring — 193 watches


In [30]:
# Highest rated movies (server-side AVG + COUNT + FILTER)
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '2', '@tmdb_id', '@rating',
    'GROUPBY', '1', '@tmdb_id',
    'REDUCE', 'AVG', '1', '@rating', 'AS', 'avg_rating',
    'REDUCE', 'COUNT', '0', 'AS', 'num_ratings',
    'FILTER', '@num_ratings >= 3',
    'SORTBY', '2', '@avg_rating', 'DESC',
    'LIMIT', '0', '10')

for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    title = r.hget(f"movie:{doc['tmdb_id']}", 'title') or doc['tmdb_id']
    print(f"  {title} — avg {float(doc['avg_rating']):.2f}★ ({doc['num_ratings']} ratings)")

  I Origins — avg 5.00★ (3 ratings)
  The Women — avg 5.00★ (3 ratings)
  Mona Lisa — avg 5.00★ (3 ratings)
  Audition — avg 5.00★ (3 ratings)
  Submarine — avg 4.90★ (5 ratings)
  Withnail & I — avg 4.88★ (4 ratings)
  The Man from Earth — avg 4.88★ (4 ratings)
  Stand and Deliver — avg 4.83★ (3 ratings)
  Purple Noon — avg 4.83★ (3 ratings)
  62128 — avg 4.83★ (3 ratings)


In [31]:
# Most popular genres (GROUPBY on TAG field)
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '1', '@genres',
    'GROUPBY', '1', '@genres',
    'REDUCE', 'COUNT', '0', 'AS', 'watch_count',
    'SORTBY', '2', '@watch_count', 'DESC',
    'LIMIT', '0', '10')

for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    print(f"  {doc['genres']} — {doc['watch_count']} watches")

  Drama — 3805 watches
  Comedy — 3146 watches
  Comedy,Romance — 2292 watches
  Drama,Romance — 1932 watches
  Action,Adventure,Sci-Fi — 1890 watches
  Comedy,Drama,Romance — 1782 watches
  Crime,Drama — 1614 watches
  Comedy,Drama — 1529 watches
  Action,Crime,Thriller — 1076 watches
  Action,Adventure,Sci-Fi,Thriller — 1041 watches


## Part 5: "Because You Watched..." — Recommendation Flow (~30 min)

This is the end-to-end recommendation pattern. All three indexes work together:

1. **Single-slot** → find what the user likes (micro)
2. **Global catalog** → find similar content via vector KNN (milliseconds)
3. **Global user index** → blend with trending data (micro-ms)

This powers experiences like "Because you watched Inception..." on streaming platforms.

### 5.1 The Pipeline


In [39]:
user_id = demo_users[0]
idx = f"idx:user:{user_id}:history"

print(f"=== Personalized Recommendations for User {user_id} ===")

# Step 1: What did they recently watch and love? (single-slot, microseconds)
print('Step 1: Most recent highly-rated movie?')
recent = r.execute_command('FT.SEARCH', idx,
    '@rating:[4 5]',
    'SORTBY', 'timestamp', 'DESC',
    'RETURN', '3', 'title', 'genres', 'tmdb_id',
    'LIMIT', '0', '1')
seed = dict(zip(recent[2][::2], recent[2][1::2]))
seed_title = seed['title']
seed_genres = seed['genres'].split(',')
seed_genre = seed_genres[0].strip()
seed_tmdb = seed['tmdb_id']
print(f"  -> '{seed_title}' (genres: {seed['genres']}, tmdb_id: {seed_tmdb})")
print()

# Step 2: What genre is THIS movie? Use it for filtering.
print(f"Step 2: Primary genre of '{seed_title}' -> {seed_genre}")
print('  (We recommend based on what they JUST watched, not all-time stats)')


=== Personalized Recommendations for User 1 ===
Step 1: Most recent highly-rated movie?
  -> 'Ever After: A Cinderella Story (1998)' (genres: Comedy,Drama,Romance, tmdb_id: 9454)

Step 2: Primary genre of 'Ever After: A Cinderella Story (1998)' -> Comedy
  (We recommend based on what they JUST watched, not all-time stats)


In [40]:
# Step 3: Fetch vector from global catalog
print(f'Step 3: Fetch vector for movie:{seed_tmdb}')
vec_blob = r_bin.hget(f'movie:{seed_tmdb}'.encode(), b'vector')

if vec_blob:
    print(f'  \u2192 {len(vec_blob)} bytes ({len(vec_blob)//4} floats)')
    print()

    # Step 4: KNN — "Since you watched X, here are similar movies"
    print(f'Step 4: Since you watched \'{seed_title}\', you might like these {seed_genre} movies:')
    knn = r_bin.execute_command('FT.SEARCH', 'idx:movies',
        f'@genres:{{{seed_genre}}}=>[KNN 11 @vector $q]',
        'PARAMS', '2', 'q', vec_blob,
        'RETURN', '3', 'title', 'genres', 'vote_average',
        'DIALECT', '2')

    print('  Recommendations:')
    for i in range(1, len(knn), 2):
        if f'movie:{seed_tmdb}'.encode() == knn[i]:
            continue
        d = dict(zip(knn[i+1][::2], knn[i+1][1::2]))
        print(f'    {d[b"title"].decode()} ({d[b"vote_average"].decode()}\u2605)')
else:
    print(f'  \u2192 movie:{seed_tmdb} not in catalog')


Step 3: Fetch vector for movie:9454
  → 3072 bytes (768 floats)

Step 4: Since you watched 'Ever After: A Cinderella Story (1998)', you might like these Comedy movies:
  Recommendations:
    Ella Enchanted (6.5★)
    That Touch of Mink (6.432★)
    The Princess Diaries (6.958★)
    Bedazzled (6.067★)
    The Prince & Me (6.2★)
    A Cinderella Story (6.6★)
    She's the Man (6.8★)
    A Life Less Ordinary (6.1★)
    The Lady Eve (7.2★)


## Part 6: Exercises (~30 min)

### Exercise 1: Filter by language
Find French (`fr`) comedies rated above 7. Hint: use `@original_language:{fr}`.

### Exercise 2: User dedup
Before recommending a movie, check if the user already watched it using their single-slot index.

### Exercise 3: Trending + Personal
Combine FT.AGGREGATE (trending movies) with the user's single-slot (exclude already watched).

### Exercise 4: Custom aggregation
Write an FT.AGGREGATE query to find the average rating per genre across all users.


In [34]:
# Exercise 1: French comedies rated 7+
# YOUR CODE HERE
results = r.execute_command('FT.SEARCH', 'idx:movies',
    '@genres:{Comedy} @original_language:{fr} @vote_average:[7 10]',
    'RETURN', '3', 'title', 'vote_average', 'original_language',
    'LIMIT', '0', '5')
print(f"French comedies rated 7+: {results[0]}")
for i in range(1, len(results), 2):
    doc = dict(zip(results[i+1][::2], results[i+1][1::2]))
    print(f"  {doc['title']} — {doc['vote_average']}★")

French comedies rated 7+: 38
  Amélie — 7.9★
  The Visitors — 7.126★
  Mon Oncle — 7.4★
  Ridicule — 7.0★
  PlayTime — 7.7★


In [35]:
# Exercise 4: Average rating per genre
# YOUR CODE HERE
results = r.execute_command('FT.AGGREGATE', 'idx:watch', '@rating:[-inf +inf]',
    'LOAD', '2', '@genres', '@rating',
    'GROUPBY', '1', '@genres',
    'REDUCE', 'AVG', '1', '@rating', 'AS', 'avg_rating',
    'REDUCE', 'COUNT', '0', 'AS', 'count',
    'FILTER', '@count >= 10',
    'SORTBY', '2', '@avg_rating', 'DESC',
    'LIMIT', '0', '10')

print("Average rating per genre (min 10 ratings):")
for row in results[1:]:
    doc = dict(zip(row[::2], row[1::2]))
    print(f"  {doc['genres']} — avg {float(doc['avg_rating']):.2f}★ ({doc['count']} ratings)")

Average rating per genre (min 10 ratings):
  Film-Noir,Mystery — avg 4.50★ (30 ratings)
  Drama,Film-Noir,Romance — avg 4.50★ (23 ratings)
  Comedy,Drama,Musical,Sci-Fi — avg 4.42★ (13 ratings)
  Animation,Drama,War — avg 4.36★ (11 ratings)
  Action,Drama,Thriller,Western — avg 4.33★ (12 ratings)
  Crime,Film-Noir,Mystery — avg 4.33★ (27 ratings)
  Crime,Thriller,War — avg 4.32★ (17 ratings)
  Film-Noir,Mystery,Thriller — avg 4.32★ (25 ratings)
  Adventure,Animation,Fantasy — avg 4.32★ (79 ratings)
  Comedy,Crime,Mystery,Romance,Thriller — avg 4.29★ (14 ratings)


## Summary

### Reusable Patterns

| Pattern | How | When |
|:--------|:----|:-----|
| Instant search / autocomplete | `@title:prefix*` | Search bars, type-ahead |
| Faceted browse | `@genre:{X} @rating:[min max]` | Catalog filtering |
| "More like this" | Vector KNN with pre-filter | Recommendations |
| "Because you watched..." | Single-slot → vector → dedup | Personalization |
| Trending / analytics | FT.AGGREGATE GROUPBY + REDUCE | Dashboards, feeds |
| Per-user data | Single-slot index | Session state, preferences |

### Latency & Tuning

- **FLAT vs HNSW**: FLAT for <10K docs (exact, fast ingest). HNSW for >10K (approximate, fast query).
- **Single-slot vs Global**: Use single-slot for user-scoped data. Global for shared data.
- **SORTABLE**: Add to NUMERIC fields you sort by — avoids runtime sort overhead.
- **RETURN**: Only return fields you need — reduces network payload.
- **LIMIT**: Always paginate — don't fetch all matches.
- **Load before index**: For bulk ingestion, HSET first then FT.CREATE (backfill is async).

### What You Built

```
Global Catalog (idx:movies)      → Browse, search, vector similarity
Single-Slot (idx:user:X:history)  → Per-user, micro, ephemeral
Global Users (idx:watch)          → Cross-user analytics, trending
```
